### Step 1: Mount the Google Drive

Remember to use GPU runtime before mounting your Google Drive. (Runtime --> Change runtime type).

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Step 2: Open the project directory

Replace `Your_Dir` with your own path.

In [ ]:
# clone the repo into Colab
!rm -rf ECE_C147A_Final
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/DysonLewis/ECE_C147A_Final.git

'''
from google.colab import files
files.upload()
!mv generic.ckpt /content/ECE_C147A_Final/Project/emg2qwerty/models/
'''

Cloning into 'ECE_C147A_Final'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 108 (delta 0), reused 0 (delta 0), pack-reused 104 (from 1)
Receiving objects: 100% (108/108), 33.52 MiB | 14.96 MiB/s, done.
Resolving deltas: 100% (9/9), done.


'\nfrom google.colab import files\nfiles.upload()\n!mv generic.ckpt /content/ECE_C147A_Final/Project/emg2qwerty/models/\n'

### Step 3: Install required packages

After installing them, Colab will require you to restart the session.

In [3]:
%cd /content/ECE_C147A_Final/Project/emg2qwerty

!mkdir data/

!cp /content/drive/MyDrive/data/* /content/ECE_C147A_Final/Project/emg2qwerty/data/

/content/ECE_C147A_Final/Project/emg2qwerty


In [ ]:
!pip install -r requirements.txt

     - 553.6 kB 9.5 MB/s 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could 

### Step 4: Start your experiments!

- Remember to download and copy the dataset to this directory: `Your_Dir/emg2qwerty/data`.
- You may now start your experiments with any scripts! Below are examples of single-user training and testing (greedy decoding).
- **There are two ways to track the logs:**
  - 1. Keep `--multirun`, and the logs will not be printed here, but they will be saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/submitit_logs/`.
  - 2. Comment out `--multirun` and the logs will be printed in this notebook, but they will not be saved.

#### Training

- The checkpoints are saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/checkpoints/`.

In [10]:
# Single-user training
%cd /content/ECE_C147A_Final/Project/emg2qwerty
!python -m emg2qwerty.train \
  user="single_user" \
  trainer.accelerator=gpu trainer.devices=1 \
  trainer.max_epochs=50 \
  batch_size=64 \
  num_workers=8 \
  optimizer.lr=0.0015 \
  +trainer.log_every_n_steps=5
  #--multirun

/content/ECE_C147A_Final/Project/emg2qwerty
[2026-03-06 01:19:15,962][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-

#### Fourier Features Training

The Nyquist verification prints how much signal power would be aliased at each downsample factor.

In [7]:
from pathlib import Path
import h5py
import numpy as np

data_dir = Path('data')
hdf5_files = list(data_dir.glob('*.hdf5'))

fs = 2000
with h5py.File(hdf5_files[0], 'r') as f:
    emg = f['emg2qwerty']['timeseries']['emg_left'][:10000, 0]

freqs = np.fft.rfftfreq(len(emg), d=1.0 / fs)
psd   = np.abs(np.fft.rfft(emg)) ** 2
total = psd.sum()

semg_pct = 100 * psd[(freqs >= 20) & (freqs <= 500)].sum() / total
print(f'Power in 20-500 Hz sEMG band: {semg_pct:.1f}% of total\n')
print(f'{"Factor":<8} {"Effective fs":<15} {"Aliased power %":<18} {"Status"}')
print('-' * 58)
for factor in [1, 2, 4, 8]:
    eff_fs  = fs // factor
    aliased = 100 * psd[freqs > (eff_fs / 2)].sum() / total
    status  = 'safe' if factor <= 2 else 'violates Nyquist'
    print(f'{factor:<8} {eff_fs:<15} {aliased:<18.2f} {status}')

data_fourier ready: 18 sessions linked
Power in 20-500 Hz sEMG band: 92.7% of total

Factor   Effective fs    Aliased power %    Status
----------------------------------------------------------
1        2000            0.00               safe
2        1000            7.34               safe
4        500             22.82              violates Nyquist
8        250             49.16              violates Nyquist


In [9]:
%cd /content/ECE_C147A_Final/Project/emg2qwerty
!python -m emg2qwerty.train \
  user="single_user" \
  transforms=fourier_features \
  dataset.root=data \
  trainer.accelerator=gpu trainer.devices=1 \
  trainer.max_epochs=50 \
  batch_size=64 \
  num_workers=8 \
  optimizer.lr=0.0015 \
  +trainer.log_every_n_steps=5
  #--multirun

/content/ECE_C147A_Final/Project/emg2qwerty
[2026-03-06 00:53:01,663][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-

#### Testing:

- Replace `Your_Path_to_Checkpoint` with your checkpoint path.

In [ ]:
import os
os.environ["HYDRA_FULL_ERROR"] = "1"

In [ ]:
%cd /content/ECE_C147A_Final/Project/emg2qwerty
!python -m emg2qwerty.train \
  user="single_user" \
  "checkpoint='/content/ECE_C147A_Final/Project/emg2qwerty/logs/2026-03-05/23-26-11/checkpoints/epoch=49-step=2000.ckpt'" \
  train=false trainer.accelerator=gpu \
  decoder=ctc_greedy

/content/ECE_C147A_Final/Project/emg2qwerty
[2026-03-05 23:59:05,166][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-